In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [16]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
import joblib
import os

df = pd.read_csv("/content/drive/MyDrive/prototipo_oliveto_pw/data/ticket_museo_le_nuove.csv")

df["text"] = df["subject"] + " " + df["description"]

In [17]:
vectorizer = joblib.load(
    "/content/drive/MyDrive/prototipo_oliveto_pw/models/tfidf_vectorizer.joblib"
)

X = vectorizer.transform(df["text"]).toarray()
X = torch.tensor(X, dtype=torch.float32)

In [18]:
priority_map = {
    "bassa": 0,
    "media": 1,
    "alta": 2
}

y = df["priority"].map(priority_map).values
y = torch.tensor(y, dtype=torch.long)

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [21]:
class TicketClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.fc(x)

model_prio = TicketClassifier(X.shape[1], 3)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_prio.parameters(), lr=0.01)

In [22]:
for epoch in range(20):
    optimizer.zero_grad()
    outputs = model_prio(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    if epoch % 5 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


Epoch 0, Loss: 1.1039
Epoch 5, Loss: 1.0284
Epoch 10, Loss: 0.9737
Epoch 15, Loss: 0.9356


In [23]:
MODEL_DIR = "/content/drive/MyDrive/prototipo_oliveto_pw/models"
os.makedirs(MODEL_DIR, exist_ok=True)

torch.save(
    model_prio.state_dict(),
    MODEL_DIR + "/priority_model.pt"
)

print("✅ priority_model.pt salvato")


✅ priority_model.pt salvato
